<a href="https://colab.research.google.com/github/NataKrj/Automated-Risk-Scoring-System/blob/main/MT_training_data%3B_H2O%3B_4_models_with_100_scenarios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libraries

In [ ]:
!pip install river scikit-learn xgboost pandas numpy matplotlib

In [ ]:
!pip install h2o

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 MB 3.3 MB/s eta 0:00:00


#Training and testing data 1-5

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

#CONFIG
CSV_WAVES = [
    "synthetic_transactions_structured1.csv",
    "synthetic_transactions_structured2.csv",
    "synthetic_transactions_structured3.csv",
    "synthetic_transactions_structured4.csv",
    "synthetic_transactions_structured5.csv",
]
TARGET_COL = "Suspicion Category"  
TEST_SIZE  = 0.20                  
RAND_SEED  = 42                     

for wave_idx, csv_path in enumerate(CSV_WAVES, start=1):
    print(f"\nProcessing {csv_path} → Wave {wave_idx}")

    df = pd.read_csv(csv_path)

    if TARGET_COL not in df.columns:
        raise KeyError(f"Target column '{TARGET_COL}' not found in {csv_path}")

    y = df[TARGET_COL]
    X = df.drop(columns=[TARGET_COL])

    # Attempt a stratified split
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, stratify=y, random_state=RAND_SEED
        )
    except ValueError as err:
        print(f"  Stratified split failed: {err}\n  Falling back to unstratified split.")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RAND_SEED
        )

    # Re‑attach target for saving
    train_df = X_train.assign(**{TARGET_COL: y_train})
    test_df  = X_test.assign(**{TARGET_COL: y_test})

    train_file = f"training_wave_{wave_idx}.csv"
    test_file  = f"testing_wave_{wave_idx}.csv"

    train_df.to_csv(train_file, index=False)
    test_df.to_csv(test_file,  index=False)

    print(f"  → saved {train_file} ({len(train_df)} rows) and {test_file} ({len(test_df)} rows)")


Processing synthetic_transactions_structured1.csv → Wave 1
  Stratified split failed: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
  Falling back to unstratified split.
  → saved training_wave_1.csv (8024 rows) and testing_wave_1.csv (2007 rows)

Processing synthetic_transactions_structured2.csv → Wave 2
  Stratified split failed: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
  Falling back to unstratified split.
  → saved training_wave_2.csv (8024 rows) and testing_wave_2.csv (2006 rows)

Processing synthetic_transactions_structured3.csv → Wave 3
  Stratified split failed: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
  Falling back to unstratified split.
  → saved training_wave_3.csv (8013 rows) and testing_wave_3.csv (200

#H2O

In [ ]:
import h2o, pandas as pd
from h2o.automl import H2OAutoML
from pathlib import Path

# ---------- CONFIG ---------------------------------------------------- #
TARGET       = "Suspicion Category"
ID_COLS      = ["Transaction_id"]
TIME_BUDGET  = 600
TRAIN_FILES  = [f"training_wave_{i}.csv" for i in range(1, 6)]
TEST_FILES   = [f"testing_wave_{i}.csv"  for i in range(1, 6)]
SAVE_DIR     = Path("models")
SAVE_DIR.mkdir(exist_ok=True)

# ---------- HELPERS --------------------------------------------------- #
def detect_constant_cols(df: pd.DataFrame) -> set[str]:
    return {c for c in df.columns if df[c].nunique(dropna=False) <= 1}

def load_csv_to_h2o(path: str, drop: set[str]) -> h2o.H2OFrame:
    pdf = pd.read_csv(path)
    pdf.drop(columns=[c for c in drop if c in pdf], inplace=True, errors="ignore")
    pdf[TARGET] = pdf[TARGET].astype("category")
    return h2o.H2OFrame(pdf)

# ---------- MAIN LOOP ------------------------------------------------- #
if __name__ == "__main__":
    h2o.init()
    cumulative_train = None
    const_cols: set[str] = {"Description"}  

    # ← NEW: prepare to capture metrics
    results = []

    for wave, (tr_file, te_file) in enumerate(zip(TRAIN_FILES, TEST_FILES), start=1):
        print(f"\n=== WAVE {wave} ===")

        # discover any new constant columns
        const_cols |= detect_constant_cols(pd.read_csv(tr_file))

        # load wave data
        wave_train = load_csv_to_h2o(tr_file, const_cols | set(ID_COLS))
        wave_test  = load_csv_to_h2o(te_file,  const_cols | set(ID_COLS))

        # grow the cumulative training set
        cumulative_train = wave_train if cumulative_train is None else cumulative_train.rbind(wave_train)
        features = [c for c in cumulative_train.col_names if c != TARGET]

        # AutoML
        aml = H2OAutoML(
            max_runtime_secs=TIME_BUDGET,
            seed=42,
            balance_classes=False,
            exclude_algos=["StackedEnsemble"],   # remove after upgrading H2O
        )
        aml.train(x=features, y=TARGET, training_frame=cumulative_train)

        # evaluate on this wave’s test set
        perf = aml.leader.model_performance(wave_test)
        print(perf)

        # ← NEW: record the leader ID and metrics
        results.append({
            "wave":     wave,
            "model_id": aml.leader.model_id,
            "logloss":  perf.logloss(),
            "mean_err": perf.mean_per_class_error(),
        })

        # save leader
        model_path = h2o.save_model(aml.leader, path=str(SAVE_DIR / f"wave_{wave}"), force=True)
        print(f"Leader saved → {model_path}")

        # free only the test frame
        h2o.remove(wave_test)

    # cleanly shut down H2O
    h2o.shutdown(prompt=False)

    # ← NEW: show a summary table of every wave’s results
    df_results = pd.DataFrame(results).sort_values("wave")
    print("\n=== Summary of Waves ===")
    print(df_results.to_string(index=False))

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.26" 2025-01-21; OpenJDK Runtime Environment (build 11.0.26+4-post-Ubuntu-1ubuntu122.04); OpenJDK 64-Bit Server VM (build 11.0.26+4-post-Ubuntu-1ubuntu122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.11/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmppnb0tfao
  JVM stdout: /tmp/tmppnb0tfao/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmppnb0tfao/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,11 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,20 days
H2O_cluster_name:,H2O_from_python_unknownUser_nrlne0
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.170 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"



=== WAVE 1 ===
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%
ModelMetricsMultinomial: xgboost
** Reported on test data. **

MSE: 8.931269077548271e-05
RMSE: 0.009450539179088287
LogLoss: 0.0007849262462639706
Mean Per-Class Error: 0.0
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
First-Time High-Risk Transactions    Large Incoming followed by Withdrawal    Normal    One-to-Many Pattern    Repeti

<ipython-input-38-6913bcd559af>:76: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown(prompt=False)
